# Geometry-aware meshing and bounded optimization

This notebook separates exact geometry, fixed-budget mesh preparation, and optional GPU reference qualification. A catalog orientation rotates the object while the source remains a +x plane wave; it is not native arbitrary-angle injection. Strict conformal references are the default. Hybrid mode can be selected explicitly in a separate study.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/fdtdmesh').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
import matplotlib.pyplot as plt
import fdtdmesh as fd
from fdtdmesh.catalog import make_geometry
from fdtdmesh.optimization import (SearchSettings, ReferenceSettings, FeasibleSettings, qualify_reference, optimize_mesh, analyze_mesh_adaptivity)
OUT = Path('artifacts/notebooks')
OUT.mkdir(parents=True, exist_ok=True)
SHAPES = ['circle']
ORIENTATIONS = [0.0]
BUDGETS = [(100, 100), (120, 120)]
RUN_GPU = False


## CPU geometry-first preparation

Each catalog geometry is installed exactly, then each requested budget is meshed with `geometry_aware`. The report prints fixed constraints and cell counts before any optional solve.

In [ ]:
prepared = {}
for shape in SHAPES:
    for orientation in ORIENTATIONS:
        sim = fd.Simulation(fmin=0.95e9, fmax=1.05e9, solver=fd.SolverSettings(dft_bins=1))
        sim.set_geometry(make_geometry(shape, scale=0.14, orientation_deg=orientation))
        key = (shape, orientation)
        prepared[key] = {}
        for cells in BUDGETS:
            sim.apply_mesh('geometry_aware', cells=cells)
            seed = sim.mesh
            prepared[key][cells] = (sim, seed)
            fig = sim.plot_geometry(mesh=True, units='wavelength', figsize=(16,14))
            fig.savefig(OUT / f'04_{shape}_{orientation:g}_{cells[0]}x{cells[1]}.png', dpi=120, bbox_inches='tight')
            print(key, cells, 'counts=', (seed.Nx, seed.Ny), 'constraints=', sim.summary().get('geometry_aware', {}))


The geometry-aware seed is a strict conformal preparation. `analyze_mesh_adaptivity` reports the fixed-index linear movement range used by the bounded feasible search; it is a diagnostic upper bound, not a guarantee of nonlinear geometry validity.

In [ ]:
for key, by_budget in prepared.items():
    for cells, (sim, seed) in by_budget.items():
        movement = analyze_mesh_adaptivity(sim, seed, projection_seconds=5.0)
        print(key, cells, 'adaptivity:', movement)


## Optional qualified reference and local search

GPU work is disabled by default. If enabled, each budget is optimized only after a subdivided strict reference qualifies. The bounded settings intentionally avoid long production searches.

In [ ]:
if RUN_GPU:
    reference_settings = ReferenceSettings(method='subdivide', factors=(2, 3, 4), rtol=.005, worst_rtol=.01)
    search_settings = SearchSettings(max_evaluations=20, max_seconds=120, max_solves=20)
    for (shape, orientation), by_budget in prepared.items():
        seed_sim, seed = by_budget[BUDGETS[0]]
        ref_dir = OUT / f'reference_{shape}_{orientation:g}'
        reference = qualify_reference(seed_sim, directory=ref_dir, settings=reference_settings, initial_mesh=seed)
        print('reference status:', reference.report.get('status'))
        if reference.report.get('status') not in ('qualified', 'pass'):
            continue
        for cells in BUDGETS:
            sim, seed = by_budget[cells]
            directory = OUT / f'optimization_{shape}_{orientation:g}_{cells[0]}x{cells[1]}'
            optimization = optimize_mesh(sim, reference, cells=cells, strategy='feasible_local', directory=directory, settings=search_settings, initial_mesh=seed)
            optimization.plot_history().savefig(directory / 'history.png', dpi=120, bbox_inches='tight')
            print('statistics:', optimization.report.get('search_statistics'))
            if optimization.best is not None:
                optimization.best.plot_geometry(mesh=True).savefig(directory / 'best_mesh.png', dpi=120, bbox_inches='tight')
